In [2]:
"""
EPL 2025/26 Season Predictor
=============================================================
Predicting EPL match outcomes and final standings using:
  1. Log ratio of pre-season squad values (home / away)
  2. Prior season points per game difference (home - away)

Model: Ordered logistic regression
Train: 2019/20 - 2024/25 (2,250 matches)
Test:  2025/26 out-of-sample (380 matches)

Data sources:
  - player_valuations.xls  : Transfermarkt player valuations
  - epl_final.xls          : EPL match results 2000/01-2024/25
  - season-2526.xls        : EPL match results 2025/26

Outputs:
  - chart1_predicted_vs_actual.png : predicted vs actual standings
  - chart3_value_vs_points.png     : squad value vs final points
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr

# =====================================================================
# CONFIGURATION
# =====================================================================

# Mapping from EPL match data team names to Transfermarkt club names
EPL_NAME_MAP = {
    "Arsenal":"Arsenal FC","Aston Villa":"Aston Villa",
    "Bournemouth":"AFC Bournemouth","Brentford":"Brentford FC",
    "Brighton":"Brighton & Hove Albion","Burnley":"Burnley FC",
    "Chelsea":"Chelsea FC","Crystal Palace":"Crystal Palace",
    "Everton":"Everton FC","Fulham":"Fulham FC",
    "Huddersfield":"Huddersfield Town","Ipswich":"Ipswich Town",
    "Leeds":"Leeds United","Leicester":"Leicester City",
    "Liverpool":"Liverpool FC","Luton":"Luton Town",
    "Man City":"Manchester City","Man United":"Manchester United",
    "Newcastle":"Newcastle United","Norwich":"Norwich City",
    "Nott'm Forest":"Nottingham Forest","Sheffield United":"Sheffield United",
    "Southampton":"Southampton FC","Tottenham":"Tottenham Hotspur",
    "Watford":"Watford FC","West Brom":"West Bromwich Albion",
    "West Ham":"West Ham United","Wolves":"Wolverhampton Wanderers",
    "Cardiff":"Cardiff City","QPR":"Queens Park Rangers",
    "Stoke":"Stoke City","Wigan":"Wigan Athletic",
    "Blackburn":"Blackburn Rovers","Bolton":"Bolton Wanderers",
    "Charlton":"Charlton Athletic","Derby":"Derby County",
    "Portsmouth":"Portsmouth FC","Reading":"Reading FC",
    "Birmingham":"Birmingham City","Blackpool":"Blackpool FC",
    "Bradford":"Bradford City","Coventry":"Coventry City",
    "Hull":"Hull City","Middlesbrough":"Middlesbrough FC",
    "Sunderland":"Sunderland AFC","Swansea":"Swansea City",
}
TM_TO_EPL    = {v: k for k, v in EPL_NAME_MAP.items()}
TM_CLUBS     = list(set(EPL_NAME_MAP.values()))

# Transfermarkt snapshot cutoff: September 1 captures full summer window
SEASON_CUTOFFS = {
    "2019/20": "2019-09-01",
    "2020/21": "2020-09-01",
    "2021/22": "2021-09-01",
    "2022/23": "2022-09-01",
    "2023/24": "2023-09-01",
    "2024/25": "2024-09-01",
    "2025/26": "2025-09-01",
}
TRAIN_SEASONS = ["2019/20","2020/21","2021/22","2022/23","2023/24","2024/25"]
TEST_SEASON   = "2025/26"
TARGET_ORDER  = ["Away Win","Draw","Home Win"]


# =====================================================================
# STEP 1: SQUAD VALUE BUILDER
# =====================================================================

def build_squad_values(vals_df):
    """
    For each season, take each player's most recent Transfermarkt
    valuation before September 1st of that year.

    Only count players whose most recent record places them at an EPL
    club — this prevents stale records (former players whose last
    known club was an EPL side) from inflating squad values.

    Returns:
        dict: {season: {epl_team_name: total_squad_value_eur}}
    """
    all_values = {}
    for season, cutoff in SEASON_CUTOFFS.items():
        # Most recent valuation per player before cutoff
        recent = (vals_df[vals_df["date"] < cutoff]
                  .sort_values("date")
                  .groupby("player_id")
                  .last()
                  .reset_index())
        # Keep only players currently at an EPL club
        epl_current = recent[
            recent["current_club_name"].isin(TM_CLUBS)
        ].copy()
        epl_current["epl_name"] = (epl_current["current_club_name"]
                                    .map(TM_TO_EPL))
        all_values[season] = (epl_current
                               .groupby("epl_name")["market_value_in_eur"]
                               .sum()
                               .to_dict())
    return all_values


# =====================================================================
# STEP 2: PPG CALCULATOR WITH PROMOTED TEAM ADJUSTMENT
# =====================================================================

def build_ppg(all_matches):
    """Compute points per game per team per season."""
    rows = []
    for _, row in all_matches.iterrows():
        h = 3 if row["FullTimeResult"]=="H" else (1 if row["FullTimeResult"]=="D" else 0)
        a = 3 if row["FullTimeResult"]=="A" else (1 if row["FullTimeResult"]=="D" else 0)
        rows.append({"Season": row["Season"], "Team": row["HomeTeam"], "pts": h})
        rows.append({"Season": row["Season"], "Team": row["AwayTeam"], "pts": a})
    ppg = (pd.DataFrame(rows)
             .groupby(["Season","Team"])["pts"]
             .mean()
             .reset_index())
    ppg.columns = ["Season","Team","PPG"]
    return ppg


def build_season_maps(ppg):
    """Build previous season map and promoted teams map."""
    seasons   = sorted(ppg["Season"].unique(),
                       key=lambda s: int(s.split("/")[0]))
    prev_map  = {seasons[i]: seasons[i-1] for i in range(1, len(seasons))}
    promo_map = {}
    for i in range(1, len(seasons)):
        curr = seasons[i]
        prev = seasons[i-1]
        promo_map[curr] = (set(ppg[ppg["Season"]==curr]["Team"]) -
                           set(ppg[ppg["Season"]==prev]["Team"]))
    return prev_map, promo_map


def calc_promoted_fallback(ppg, promo_map):
    """
    Historical average first-season PPG for promoted teams.
    Assigned to any team promoted from the Championship —
    their Championship PPG is not comparable to EPL PPG.
    """
    vals = []
    for s in TRAIN_SEASONS:
        for t in promo_map.get(s, set()):
            v = ppg[(ppg["Season"]==s) & (ppg["Team"]==t)]["PPG"]
            if len(v):
                vals.append(v.values[0])
    return np.mean(vals)


# =====================================================================
# STEP 3: FEATURE ENGINEERING
# =====================================================================

def add_features(df, season_values, ppg_lookup, prev_map,
                 promo_map, fallback):
    """
    For each match, compute:
      LogRatio  = log(home squad value / away squad value)
      PPG_diff  = home prior PPG - away prior PPG

    Promoted teams receive the historical average promoted-team PPG
    instead of their Championship PPG.

    Matches where either team has no squad value are dropped.
    """
    rows = []
    for _, row in df.iterrows():
        sv = season_values.get(row["Season"], {})
        hv = sv.get(row["HomeTeam"], None)
        av = sv.get(row["AwayTeam"], None)
        if hv is None or av is None or hv == 0 or av == 0:
            continue

        def prior_ppg(team):
            prev = prev_map.get(row["Season"])
            if prev is None:
                return fallback
            if team in promo_map.get(row["Season"], set()):
                return fallback
            return ppg_lookup.get((prev, team), fallback)

        rows.append({
            "Season":    row["Season"],
            "HomeTeam":  row["HomeTeam"],
            "AwayTeam":  row["AwayTeam"],
            "HomeValue": hv,
            "AwayValue": av,
            "LogRatio":  np.log(hv / av),
            "PPG_diff":  prior_ppg(row["HomeTeam"]) -
                         prior_ppg(row["AwayTeam"]),
            "Result":    row["FullTimeResult"],
            "Outcome":   2 if row["FullTimeResult"]=="H"
                         else (1 if row["FullTimeResult"]=="D" else 0),
            "HomeGoals": row["FullTimeHomeGoals"],
            "AwayGoals": row["FullTimeAwayGoals"],
        })
    return pd.DataFrame(rows)


# =====================================================================
# STEP 4: STANDINGS SIMULATOR
# =====================================================================

def simulate_standings(test_df, pred_arr):
    """
    Convert match-level probabilities to season standings.
    Expected points per match = 3*P(win) + 1*P(draw) + 0*P(loss)
    Summed across all 38 matches per team.
    """
    df = test_df.copy().reset_index(drop=True)
    df[TARGET_ORDER] = pred_arr
    df["home_exp"] = 3*df["Home Win"] + 1*df["Draw"]
    df["away_exp"] = 3*df["Away Win"] + 1*df["Draw"]
    df["home_act"] = df["Result"].map({"H":3,"D":1,"A":0})
    df["away_act"] = df["Result"].map({"A":3,"D":1,"H":0})
    exp = (df.groupby("HomeTeam")["home_exp"].sum() +
           df.groupby("AwayTeam")["away_exp"].sum()).sort_values(ascending=False)
    act = (df.groupby("HomeTeam")["home_act"].sum() +
           df.groupby("AwayTeam")["away_act"].sum()).sort_values(ascending=False)
    return exp, act


# =====================================================================
# MAIN
# =====================================================================

if __name__ == "__main__":

    # ------------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------------
    print("Loading data...")
    vals = pd.read_csv("player_valuations.csv")
    vals["date"] = pd.to_datetime(vals["date"])

    epl = pd.read_csv("epl_final.csv")
    s26 = pd.read_csv("season-2526.csv")
    s26 = s26.rename(columns={
        "Date":  "MatchDate",
        "FTR":   "FullTimeResult",
        "FTHG":  "FullTimeHomeGoals",
        "FTAG":  "FullTimeAwayGoals",
    })
    s26["Season"] = "2025/26"

    common = ["Season","MatchDate","HomeTeam","AwayTeam",
              "FullTimeHomeGoals","FullTimeAwayGoals","FullTimeResult"]
    all_matches = pd.concat([epl[common], s26[common]], ignore_index=True)

    # ------------------------------------------------------------------
    # Build squad values
    # ------------------------------------------------------------------
    print("Building squad values (Sep 1 snapshot per season)...")
    all_values = build_squad_values(vals)

    # ------------------------------------------------------------------
    # Build PPG and promoted team maps
    # ------------------------------------------------------------------
    print("Building PPG and season maps...")
    ppg                  = build_ppg(all_matches)
    prev_map, promo_map  = build_season_maps(ppg)
    fallback             = calc_promoted_fallback(ppg, promo_map)
    ppg_lookup           = ppg.set_index(["Season","Team"])["PPG"].to_dict()
    print(f"  Promoted team fallback PPG: {fallback:.4f}")

    # ------------------------------------------------------------------
    # Build feature datasets
    # ------------------------------------------------------------------
    print("Engineering features...")
    train_df = add_features(
        all_matches[all_matches["Season"].isin(TRAIN_SEASONS)],
        all_values, ppg_lookup, prev_map, promo_map, fallback)
    test_df  = add_features(
        all_matches[all_matches["Season"]==TEST_SEASON],
        all_values, ppg_lookup, prev_map, promo_map, fallback)

    print(f"  Training matches: {len(train_df)}")
    print(f"  Test matches:     {len(test_df)}")

    # ------------------------------------------------------------------
    # Train ordered logistic regression
    # ------------------------------------------------------------------
    print("\nTraining ordered logistic regression...")
    train_df["Outcome_ord"] = pd.Categorical(
        train_df["Outcome"].map({0:"Away Win",1:"Draw",2:"Home Win"}),
        categories=TARGET_ORDER, ordered=True)

    model  = OrderedModel(train_df["Outcome_ord"],
                          train_df[["LogRatio","PPG_diff"]],
                          distr="logit")
    result = model.fit(method="bfgs", disp=False)

    print("\n--- Model Summary ---")
    print(result.summary())

    # ------------------------------------------------------------------
    # Predictions
    # ------------------------------------------------------------------
    pred_train = result.predict(train_df[["LogRatio","PPG_diff"]])
    pred_train.columns = TARGET_ORDER
    pred_test  = result.predict(test_df[["LogRatio","PPG_diff"]])
    pred_test.columns  = TARGET_ORDER

    y_true_train = train_df["Outcome"].map({0:"Away Win",1:"Draw",2:"Home Win"})
    y_true_test  = test_df["Outcome"].map({0:"Away Win",1:"Draw",2:"Home Win"})
    y_pred_train = [TARGET_ORDER[i] for i in pred_train.values.argmax(axis=1)]
    y_pred_test  = [TARGET_ORDER[i] for i in pred_test.values.argmax(axis=1)]

    train_acc = accuracy_score(y_true_train, y_pred_train)
    test_acc  = accuracy_score(y_true_test,  y_pred_test)
    naive_acc = train_df["Result"].value_counts().max() / len(train_df)

    print(f"\n--- Accuracy ---")
    print(f"In-sample accuracy:     {train_acc:.4f}  ({train_acc*100:.1f}%)")
    print(f"Out-of-sample accuracy: {test_acc:.4f}   ({test_acc*100:.1f}%)")
    print(f"Naive baseline:         {naive_acc:.4f}   ({naive_acc*100:.1f}%)")

    # ------------------------------------------------------------------
    # Brier scores
    # ------------------------------------------------------------------
    print("\n--- Brier Skill Scores (out-of-sample) ---")
    actual   = test_df["Outcome"].values
    pred_arr = pred_test.values
    for i, label in enumerate(TARGET_ORDER):
        ab  = (actual == i).astype(int)
        nb  = np.mean((ab.mean() - ab)**2)
        bs  = np.mean((pred_arr[:, i] - ab)**2)
        bss = 1 - bs / nb
        print(f"  {label:<12}: Brier={bs:.4f}  Naive={nb:.4f}  BSS={bss:+.4f}")

    # ------------------------------------------------------------------
    # Simulated standings
    # ------------------------------------------------------------------
    exp_pts, act_pts = simulate_standings(test_df, pred_arr)
    pred_rank = {t: i+1 for i, t in enumerate(exp_pts.index)}
    act_rank  = {t: i+1 for i, t in enumerate(act_pts.index)}
    teams     = list(act_pts.index)

    rho, pval = spearmanr(
        [pred_rank.get(t, 20) for t in teams],
        [act_rank[t] for t in teams])

    print(f"\n--- 2025/26 Predicted vs Actual Standings ---")
    print(f"{'Team':<22} {'Actual':>7} {'Predicted':>10} {'Act Pts':>9}")
    print("=" * 53)
    for team in act_pts.index:
        print(f"{team:<22} {act_rank[team]:>7} "
              f"{pred_rank.get(team,'?'):>10} "
              f"{act_pts[team]:>9.0f}")
    print(f"\nSpearman rank correlation: {rho:.4f} (p={pval:.4f})")

    sv_2526 = all_values[TEST_SEASON]

    # ==================================================================
    # CHART 1: Predicted vs Actual Standings
    # x = actual position, y = predicted position
    # Above diagonal = overperformed model (green)
    # Below diagonal = underperformed model (orange)
    # ==================================================================
    label_offsets_1 = {
        "Arsenal":        ( 6,  4), "Man City":       ( 6, -8),
        "Man United":     ( 6,  4), "Aston Villa":    ( 6, -8),
        "Liverpool":      ( 6,  4), "Bournemouth":    ( 6, -8),
        "Sunderland":     (-75, 4), "Brighton":       ( 6,  4),
        "Brentford":      ( 6, -8), "Chelsea":        ( 6,  4),
        "Fulham":         ( 6, -8), "Newcastle":      ( 6,  4),
        "Everton":        ( 6, -8), "Leeds":          ( 6,  4),
        "Crystal Palace": ( 6,  4), "Nott'm Forest":  ( 6, -8),
        "Tottenham":      ( 6,  4), "West Ham":       ( 6, -8),
        "Burnley":        ( 6,  4), "Wolves":         ( 6, -8),
    }

    fig, ax = plt.subplots(figsize=(9, 9))

    for team in teams:
        pr    = pred_rank.get(team, 20)
        ar    = act_rank[team]
        diff  = pr - ar
        color = ("#1baf7a" if diff > 2
                 else ("#eb6834" if diff < -2
                 else "#2a78d6"))
        ax.scatter(ar, pr, color=color, s=90, zorder=3,
                   edgecolors="white", linewidths=0.6)
        dx, dy = label_offsets_1.get(team, (6, 4))
        ax.annotate(team, xy=(ar, pr),
                    xytext=(dx, dy), textcoords="offset points",
                    fontsize=8, color=color, fontweight="medium")

    # Perfect prediction diagonal
    ax.plot([1,20],[1,20], color="#aaa", linewidth=1,
            linestyle=":", zorder=1)

    # Shading
    ax.fill_between([1,20],[1,20],[1,1],
                    alpha=0.04, color="#1baf7a")
    ax.fill_between([1,20],[1,20],[20,20],
                    alpha=0.04, color="#eb6834")

    ax.set_xlim(0.5, 20.5)
    ax.set_ylim(0.5, 20.5)
    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.set_xlabel("Actual final position", fontsize=9)
    ax.set_ylabel("Predicted final position", fontsize=9)
    ax.set_title(
        "EPL 2025/26 — Predicted vs Actual Final Standings\n"
        f"Ordered logit: log(squad value ratio) + prior season PPG diff"
        f"  ·  Spearman ρ = {rho:.3f}",
        fontsize=11, pad=12)
    ax.set_xticks(range(1, 21))
    ax.set_yticks(range(1, 21))
    ax.tick_params(labelsize=8)

    legend_elements = [
        mpatches.Patch(color="#1baf7a",
                       label="Overperformed model (finished higher than predicted)"),
        mpatches.Patch(color="#eb6834",
                       label="Underperformed model (finished lower than predicted)"),
        mpatches.Patch(color="#2a78d6",
                       label="Close to prediction (≤2 places)"),
    ]
    ax.legend(handles=legend_elements, fontsize=8,
              frameon=True, framealpha=0.9, loc="lower right")
    ax.grid(color="#e1e0d9", linewidth=0.4, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top","right"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.savefig("chart1_predicted_vs_actual.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("\nSaved: chart1_predicted_vs_actual.png")

    # ==================================================================
    # CHART 3: Pre-season squad value vs Final points
    # Colour coded by finishing position tier
    # ==================================================================
    teams_plot = [t for t in teams if t in sv_2526]
    values_m   = [sv_2526[t]/1e6 for t in teams_plot]
    points_    = [act_pts[t] for t in teams_plot]

    def pos_color(team):
        ar = act_rank.get(team, 20)
        if ar <= 4:  return "#2a78d6"
        if ar <= 7:  return "#1baf7a"
        if ar <= 14: return "#eda100"
        return "#eb6834"

    colors = [pos_color(t) for t in teams_plot]
    r      = np.corrcoef(values_m, points_)[0,1]

    label_offsets_3 = {
        "Arsenal":        (  5, -10), "Man City":        (  5,   5),
        "Man United":     ( -5, -10), "Aston Villa":     (  5,   5),
        "Liverpool":      ( -5, -10), "Bournemouth":     (  5,   5),
        "Sunderland":     (  5,   5), "Brighton":        (  5, -10),
        "Brentford":      ( -5,   8), "Chelsea":         (  5, -10),
        "Fulham":         (-65, -10), "Newcastle":       (  5,   5),
        "Everton":        (  5,   5), "Leeds":           (  5,   5),
        "Crystal Palace": (  5, -10), "Nott'm Forest":   (  5,   5),
        "Tottenham":      (  5,   5), "West Ham":        (-70,   5),
        "Burnley":        (  5,   5), "Wolves":          (  5, -10),
    }

    fig, ax = plt.subplots(figsize=(11, 8))
    ax.scatter(values_m, points_, c=colors, s=85,
               zorder=3, alpha=0.85,
               edgecolors="white", linewidths=0.6)

    for team, val, pts in zip(teams_plot, values_m, points_):
        dx, dy = label_offsets_3.get(team, (5, 5))
        ax.annotate(team, xy=(val, pts),
                    xytext=(dx, dy), textcoords="offset points",
                    fontsize=8, color="#333", fontweight="medium")

    z  = np.polyfit(values_m, points_, 1)
    xr = np.linspace(min(values_m)-30, max(values_m)+30, 100)
    ax.plot(xr, np.poly1d(z)(xr), color="#898781",
            linewidth=1.5, linestyle="--", zorder=2)

    ax.set_xlabel("Pre-season squad value (€m, Transfermarkt)", fontsize=9)
    ax.set_ylabel("Final league points (2025/26)", fontsize=9)
    ax.set_title(
        f"EPL 2025/26 — Pre-season squad value vs final points\n"
        f"r = {r:.3f}  ·  Money explains some but not all of the final table",
        fontsize=11, pad=12)

    legend_elements = [
        mpatches.Patch(color="#2a78d6", label="Champions League (top 4)"),
        mpatches.Patch(color="#1baf7a", label="European places (5–7)"),
        mpatches.Patch(color="#eda100", label="Mid-table (8–14)"),
        mpatches.Patch(color="#eb6834", label="Bottom half (15–20)"),
    ]
    ax.legend(handles=legend_elements, fontsize=8,
              frameon=True, framealpha=0.9, loc="upper left")
    ax.grid(color="#e1e0d9", linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top","right"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.savefig("chart3_value_vs_points.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: chart3_value_vs_points.png")

    print("\nDone.")

Loading data...
Building squad values (Sep 1 snapshot per season)...
Building PPG and season maps...
  Promoted team fallback PPG: 0.8299
Engineering features...
  Training matches: 2250
  Test matches:     380

Training ordered logistic regression...

--- Model Summary ---
                             OrderedModel Results                             
Dep. Variable:            Outcome_ord   Log-Likelihood:                -2225.9
Model:                   OrderedModel   AIC:                             4460.
Method:            Maximum Likelihood   BIC:                             4483.
Date:                Wed, 19 Aug 2026                                         
Time:                        02:31:31                                         
No. Observations:                2250                                         
Df Residuals:                    2246                                         
Df Model:                           2                                         
              